# FitCheck Open Model Bake-off
Run on a Kaggle or Colab GPU from the FitCheck repo root. Turn on internet access for dependency and weight downloads. Use a fresh runtime per model to release VRAM and avoid conflicting dependencies. Keep the input manifest and outputs private and delete them within 24 hours or on request. Commit aggregate scores only. The actual run is **not run** until approved inputs and GPU runs exist.

- [FASHN VTON v1.5](https://github.com/fashn-AI/fashn-vton-1.5) uses Apache-2.0.
- [CatVTON](https://github.com/Zheng-Chong/CatVTON) and [IDM-VTON](https://github.com/yisol/IDM-VTON) use CC BY-NC-SA 4.0 for the model releases. Use for the noncommercial hackathon demo under their terms.

Use four garment-only kurta photos and three adult team photos with written consent, or eligible research dataset inputs. Match all four garments with all three people on all three models. In photos/bakeoff.json use people entries with file and authorized_adult_photo, and garments entries with file and description. Paths are relative to the manifest.


In [ ]:
from pathlib import Path
from subprocess import check_call
import sys
import torch

assert Path('bakeoff.py').is_file(), 'Run from the FitCheck repository root'
assert torch.cuda.is_available(), 'Enable the notebook GPU'
check_call([sys.executable, '-m', 'pip', 'install', 'fastapi', 'uvicorn', 'python-multipart', 'httpx',
            'mediapipe', 'pillow', 'openpyxl', 'gradio_client'])
check_call(['curl','-fsSL','-o','selfie_multiclass_256x256.tflite',
'https://storage.googleapis.com/mediapipe-models/image_segmenter/selfie_multiclass_256x256/float32/1/selfie_multiclass_256x256.tflite'])


## Install a model
FASHN uses the setup cell below. For CatVTON clone the official `edited` branch and install its official requirements. Set PYTHONPATH to the CatVTON checkout when starting FitCheck's gpu_worker. For IDM-VTON follow the official Gradio setup in a separate environment on this host and launch the official demo on loopback port 7861. Change `api_name="tryon"` on its button registration if the checked-out app does not expose /tryon. Never bind its Gradio app to a public interface. Manage its upload cache as private temporary storage and clear it on request or within 24 hours. It is a bake-off dependency, not the recommended public serving worker.


In [ ]:
import os
import secrets
from subprocess import Popen, DEVNULL

MODEL = 'fashn'  # catvton or idm after their own setup
if MODEL == 'fashn':
    if not Path('models/fashn-vton-1.5').exists():
        check_call(['git','clone','https://github.com/fashn-AI/fashn-vton-1.5','models/fashn-vton-1.5'])
    checkout = Path('models/fashn-vton-1.5').resolve()
    check_call([sys.executable,'-m','pip','install',str(checkout)])
    weights = checkout / 'weights'
    check_call([sys.executable,str(checkout/'scripts/download_weights.py'),'--weights-dir',str(weights)],cwd=checkout)
    os.environ['FASHN_WEIGHTS'] = str(weights)
elif MODEL == 'catvton':
    checkout = Path('models/CatVTON').resolve()
    assert checkout.is_dir()
    os.environ['PYTHONPATH'] = str(checkout)
os.environ['FITCHECK_GPU_MODEL'] = MODEL
os.environ.setdefault('TRYON_HTTP_TOKEN',secrets.token_urlsafe(32))
worker = Popen([sys.executable,'-m','uvicorn','gpu_worker:app','--host','127.0.0.1','--port','7860','--no-access-log'],
               stdout=DEVNULL, stderr=DEVNULL)


In [ ]:
import httpx
from time import sleep
for _ in range(30):
    try:
        httpx.get('http://127.0.0.1:7860/openapi.json',timeout=3).raise_for_status()
        break
    except httpx.HTTPError:
        sleep(1)
else:
    raise RuntimeError('GPU worker did not start')
try:
    check_call([sys.executable,'bakeoff.py',MODEL,'--manifest','photos/bakeoff.json'])
finally:
    worker.terminate()
    worker.wait()


In [ ]:
import csv
from IPython.display import display
from PIL import Image, ImageOps
output = Path('renders/bakeoff')
for i in range(1,4):
    for j in range(1,5):
        paths = [output/f'{model}-{i}-{j}.png' for model in ('catvton','idm','fashn')]
        shown = [(path.name,ImageOps.contain(Image.open(path).convert('RGB'),(280,440))) for path in paths if path.exists()]
        if shown:
            montage = Image.new('RGB',(280*len(shown),440),'#ece5e2')
            for n, (name,image) in enumerate(shown):
                print(name)
                montage.paste(image,(280*n,0))
            display(montage)
for path in output.glob('*.csv'):
    with path.open() as stream:
        for row in csv.DictReader(stream):
            print(row)


## Record the decision
Enter human print-fidelity scores (1–5) in each CSV after comparing with the original garment. Record runtime GPU, model commit, warm-up and seed. Compare failures and changed outside-mask pixels before and after compositing. Choose using the completed table, prioritising preservation and print fidelity, then latency. **Winner: not run.** FASHN is the initial deploy candidate based on its open licence and documented single-GPU runtime.
